# US-07 - Regresión logística multinomial (M1): baseline y mejora

**Objetivo y alcance.** Entrenar la regresión logística multinomial (softmax) sobre el Pipeline compartido del proyecto (US-05), primero como *baseline* y después mejorada, para obtener una configuración final comparable con los otros tres modelos en US-11.

**Reglas del proyecto aplicadas:**
- Entrenamiento **solo** con `data/train_70.csv`; `val_15.csv` solo para evaluar. `test_15.csv` **no se carga ni se usa** (D-012).
- Imputación (D-011, aprobada), encoding (D-014, aprobada) y escalado viven **dentro** del Pipeline; nada se aplica a mano a los datos (D-013, ítem 3).
- Semilla única importada de `src.config.RANDOM_STATE`; no se crean semillas nuevas.
- Métrica principal: **macro-F1** en validación (D-010). Límite de sobreajuste: `accuracy(train) - accuracy(val) < 5` puntos porcentuales, en absoluto; exactamente 5.00 **no** cumple (D-010, respuesta del profesor).
- Toda cifra reportada debe venir de una ejecución registrada (AGENTS.md §5): las salidas de este notebook provienen de ejecuciones reales y las métricas quedan registradas en `results/logistic_regression.csv` (nada inventado).

**Estado de dependencias:** D-011 (imputación, aprobada por Brian el 2026-11-13) y **D-014 (encoding, aprobada por Brian el 2026-11-16 vía PR #61, ítem 2 corregido)** están conectadas por parámetros a través de `build_preprocessing()` (D-014, ítem 5: el módulo compartido no se modifica). El `StandardScaler` es paso propio de M1 (D-014, ítem 6). **US-05 fase 2** (los defaults del módulo compartido) sigue sin integrarse en `develop`, así que **todas las métricas de este notebook se marcan PROVISIONALES** hasta que aterrice y se re-ejecute (D-013, Consequences).

## 1. Importaciones y configuración

Importamos la semilla del proyecto desde `src.config` y el modelo reutilizable de `src.model_logreg`, que a su vez usa el preprocesamiento compartido de `src.preprocessing` (`build_preprocessing`). No duplicamos ninguna lógica de `src/`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)

from src.config import RANDOM_STATE
from src.model_logreg import CANDIDATE_CONFIGS, build_logreg_pipeline
from src.preprocessing import make_features_target

warnings.filterwarnings("ignore", message=".*penalty.*deprecated.*")

print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__}")
print(f"RANDOM_STATE (src.config): {RANDOM_STATE}")


Python: 3.12.10 | pandas: 3.1.1
RANDOM_STATE (src.config): 42


## 2. Carga de datos

Cargamos **únicamente** las particiones compartidas permitidas para entrenar y validar. `test_15.csv` se reserva para la evaluación final (US-11) y no aparece en este notebook. Ejecutar desde la raíz del repositorio.

In [2]:
DATA = Path("data")
train_df = pd.read_csv(DATA / "train_70.csv")
val_df = pd.read_csv(DATA / "val_15.csv")

print(f"train_70: {train_df.shape} | val_15: {val_df.shape}")
print(train_df.head().to_string())

train_70: (5647, 11) | val_15: (1210, 11)
       ID  Gender Ever_Married  Age Graduated Profession  Work_Experience Spending_Score  Family_Size  Var_1 Segmentation
0  460968    Male          Yes   30        No     Doctor             11.0        Average          2.0  Cat_3            A
1  461619    Male          Yes   53       Yes  Executive              NaN           High          3.0  Cat_6            C
2  467217  Female          Yes   79       Yes     Lawyer              NaN           High          2.0  Cat_6            C
3  459338    Male          Yes   88        No  Executive              NaN            Low          NaN  Cat_6            D
4  465826    Male          Yes   45        No  Executive              3.0            Low          9.0  Cat_4            D


## 3. Separación de variables predictoras y target

`make_features_target` (US-05) selecciona las features por nombre desde `src.config`, excluye `ID` (nunca es feature) y devuelve el target aparte, que nunca entra en el preprocesamiento. El dataframe de entrada no se modifica.

In [3]:
X_train, y_train = make_features_target(train_df)
X_val, y_val = make_features_target(val_df)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")
print(f"Features ({len(X_train.columns)}): {list(X_train.columns)}")
print(f"Clases del target: {sorted(y_train.unique())}")

X_train: (5647, 9) | X_val: (1210, 9)
Features (9): ['Age', 'Work_Experience', 'Family_Size', 'Ever_Married', 'Graduated', 'Gender', 'Profession', 'Var_1', 'Spending_Score']
Clases del target: ['A', 'B', 'C', 'D']


## 4. Baseline: DummyClassifier

Referencia mínima de D-010: `DummyClassifier(strategy="most_frequent")` predice siempre la clase mayoritaria (D). Todo modelo real debe superar claramente su macro-F1 en validación. Esta celda **no depende de D-014** y puede ejecutarse ya: el baseline no necesita encoding porque ignora las features.

In [4]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

for name, X, y in (("train_70", X_train, y_train), ("val_15", X_val, y_val)):
    pred = baseline.predict(X)
    print(
        f"{name}: accuracy={accuracy_score(y, pred):.4f} | "
        f"macro_f1={f1_score(y, pred, average='macro'):.4f}"
    )

train_70: accuracy=0.2810 | macro_f1=0.1097
val_15: accuracy=0.2810 | macro_f1=0.1097


## 5. Pipeline de Logistic Regression (estructura)

El pipeline de M1 concatena: (1) el preprocesamiento compartido con los imputers aprobados de D-011 y los encoders aprobados de D-014 (conectados por parámetros desde `src.model_logreg`), (2) el `StandardScaler` específico de regresión logística (D-014, ítem 6: después del preprocesamiento compartido, antes del estimador) y (3) `LogisticRegression` multinomial (por defecto en scikit-learn 1.9). La celda marca `D014_APPROVED = True` (D-014 aprobada por Brian el 2026-11-16, PR #61).

In [5]:
from src import preprocessing

print("Slots del módulo compartido (US-05 sigue en fase 1; este modelo los conecta por parámetros):")
print(preprocessing.PENDING_DECISIONS)

# D-014 aprobada por Brian el 2026-11-16 (PR #61): los encoders se conectan
# por parámetros desde src.model_logreg (D-014, ítem 5); el módulo compartido
# no se modifica. US-05 fase 2 no está integrada aún -> métricas PROVISIONALES.
D014_APPROVED = True

model = build_logreg_pipeline()  # baseline: C=1.0, penalty=l2, solver=lbfgs
print(f"\nD014_APPROVED = {D014_APPROVED}")
print(f"Pasos del Pipeline: {[s[0] for s in model.steps]}")

Slots del módulo compartido (US-05 sigue en fase 1; este modelo los conecta por parámetros):
{'imputation': 'D-011 (Approved by Brian, 2026-11-13) — strategies approved, implementation deferred to phase 2', 'encoding': 'not yet recorded; candidate identifier D-014, pending team approval'}

D014_APPROVED = True
Pasos del Pipeline: ['preprocessing', 'scaler', 'classifier']


## 6. Entrenamiento y evaluación

Se entrena **solo con `train_70`** y se evalúa en `val_15` con accuracy, macro-F1 y el reporte por clase. La imputación, el encoding y el escalado se ajustan exclusivamente con datos de entrenamiento; `val_15` solo se transforma (sin fuga de datos). Tras el ajuste se comprueba el ancho real de la matriz transformada. Mientras US-05 fase 2 no se integre en `develop`, las métricas quedan marcadas como **PROVISIONALES**.

In [6]:
model.fit(X_train, y_train)

width = model.named_steps["preprocessing"].transform(X_train).shape[1]
feature_names = model.named_steps["preprocessing"].get_feature_names_out()
print(f"Ancho de la matriz tras el preprocesamiento (ajustado en train_70): {width} columnas")
print(f"Ejemplos de nombres: {list(feature_names[:8])} ... {list(feature_names[-3:])}")

for name, X, y in (("train_70", X_train, y_train), ("val_15", X_val, y_val)):
    pred = model.predict(X)
    print(
        f"--- {name} --- accuracy={accuracy_score(y, pred):.4f} | "
        f"macro_f1={f1_score(y, pred, average='macro'):.4f}"
    )

print(classification_report(y_val, model.predict(X_val), digits=4))


Ancho de la matriz tras el preprocesamiento (ajustado en train_70): 28 columnas
Ejemplos de nombres: ['Age', 'Work_Experience', 'Family_Size', 'missingindicator_Work_Experience', 'missingindicator_Family_Size', 'Ever_Married_Yes', 'Graduated_Yes', 'Gender_Male'] ... ['Spending_Score_Average', 'Spending_Score_High', 'Spending_Score_Low']
--- train_70 --- accuracy=0.5128 | macro_f1=0.4894
--- val_15 --- accuracy=0.5058 | macro_f1=0.4797


              precision    recall  f1-score   support

           A     0.4007    0.4088    0.4047       296
           B     0.3669    0.2222    0.2768       279
           C     0.4957    0.5797    0.5344       295
           D     0.6548    0.7588    0.7030       340

    accuracy                         0.5058      1210
   macro avg     0.4795    0.4924    0.4797      1210
weighted avg     0.4874    0.5058    0.4906      1210



## 7. Análisis de sobreajuste (D-010)

El criterio oficial compara accuracy de train y validación como **diferencia absoluta** en puntos porcentuales: cumple si es **estrictamente menor que 5 pp**; exactamente 5.00 no cumple. El gap de macro-F1 se reporta como informativo.

In [7]:
acc_train = accuracy_score(y_train, model.predict(X_train))
acc_val = accuracy_score(y_val, model.predict(X_val))
gap_pp = (acc_train - acc_val) * 100

print(f"accuracy(train_70) = {acc_train:.4f}")
print(f"accuracy(val_15)   = {acc_val:.4f}")
print(f"diferencia         = {gap_pp:.2f} pp -> "
      f"{'CUMPLE' if gap_pp < 5.0 else 'NO CUMPLE'} el límite de D-010 (< 5 pp)")

accuracy(train_70) = 0.5128
accuracy(val_15)   = 0.5058
diferencia         = 0.71 pp -> CUMPLE el límite de D-010 (< 5 pp)


## 8. Comparación de experimentos y selección con D-015 (mejora)

Las configuraciones candidatas (`CANDIDATE_CONFIGS` en `src/model_logreg.py`: fuerza de regularización `C`, `l1`/`l2` con `saga`, `class_weight`) se comparan con **validación cruzada estratificada de 5 particiones sobre `train_70`**, pasando el **Pipeline completo** a `cross_validate` (D-010, ítem 6; D-015, ítem 2): así la imputación, el encoding y el escalado se ajustan dentro de cada fold y no hay fuga de información. La media y la desviación típica de accuracy y macro-F1 se calculan sobre los **cinco scores de fold** (NumPy, `ddof=0`), no sobre predicciones OOF agrupadas.

La configuración se elige **solo con CV**, aplicando D-015: (1) se descartan las configuraciones con train - CV accuracy gap ≥ 5 pp; (2) se toma la de mayor macro-F1 medio de CV; (3) se considera empatada toda configuración admisible a menos de un error estándar (`std / sqrt(5)`) de la mejor; (4) entre las empatadas se elige la de menor train - CV accuracy gap. `val_15` no participa en la selección: se usa después, una sola vez, para la comprobación oficial de D-010. Todas las cifras son **PROVISIONALES** hasta que US-05 fase 2 se integre en `develop`.


In [8]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate

CLASSES = ["A", "B", "C", "D"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    **{
        f"f1_{label}": make_scorer(
            f1_score, labels=[label], average="macro", zero_division=0
        )
        for label in CLASSES
    },
}

rows = []
for cfg in CANDIDATE_CONFIGS:
    pipe = build_logreg_pipeline(
        C=cfg["C"],
        penalty=cfg["penalty"],
        solver=cfg["solver"],
        class_weight=cfg["class_weight"],
    )
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True
    )
    acc = scores["test_accuracy"]
    f1 = scores["test_macro_f1"]
    row = {
        "config": cfg["config"],
        "cv_accuracy_mean": acc.mean(),
        "cv_accuracy_std": acc.std(),
        "cv_macro_f1_mean": f1.mean(),
        "cv_macro_f1_std": f1.std(),
        "train_cv_acc_gap_pp": (scores["train_accuracy"].mean() - acc.mean()) * 100,
    }
    for label in CLASSES:
        row[f"cv_f1_{label}"] = scores[f"test_f1_{label}"].mean()
    rows.append(row)

cv_summary = pd.DataFrame(rows)
cv_summary["admissible"] = cv_summary["train_cv_acc_gap_pp"] < 5.0
admissible = cv_summary[cv_summary["admissible"]].copy()
if admissible.empty:
    selected_cv = cv_summary.sort_values(
        ["train_cv_acc_gap_pp", "cv_macro_f1_mean"], ascending=[True, False]
    ).iloc[0]
    selection_note = "ninguna configuración cumple el corte CV (< 5 pp); se aplica D-015 ítem 6"
else:
    best = admissible.loc[admissible["cv_macro_f1_mean"].idxmax()]
    standard_error = best["cv_macro_f1_std"] / np.sqrt(cv.n_splits)
    threshold = best["cv_macro_f1_mean"] - standard_error
    tied = admissible[admissible["cv_macro_f1_mean"] >= threshold].copy()
    selected_cv = tied.sort_values(
        ["train_cv_acc_gap_pp", "cv_macro_f1_mean"], ascending=[True, False]
    ).iloc[0]
    selection_note = (
        f"mejor macro-F1 CV={best['cv_macro_f1_mean']:.6f}; "
        f"SE={standard_error:.6f}; umbral={threshold:.6f}; "
        f"empatadas={', '.join(tied['config'].tolist())}"
    )

print(cv_summary.sort_values("cv_macro_f1_mean", ascending=False).to_string(index=False))
print(f"\nD-015: {selection_note}")
print(f"Configuración seleccionada por D-015: {selected_cv['config']}")


         config  cv_accuracy_mean  cv_accuracy_std  cv_macro_f1_mean  cv_macro_f1_std  train_cv_acc_gap_pp  cv_f1_A  cv_f1_B  cv_f1_C  cv_f1_D  admissible
C=1_l2_balanced          0.499024         0.009642          0.485673         0.010076             1.394710 0.443625 0.302905 0.539134 0.657026        True
       C=0.1_l2          0.501857         0.014946          0.479499         0.017332             1.377112 0.436092 0.270507 0.546174 0.665222        True
    C=1_l1_saga          0.501503         0.014235          0.479009         0.016764             1.416939 0.434050 0.270611 0.546527 0.664850        True
        C=10_l2          0.501326         0.012772          0.478991         0.015005             1.461180 0.434608 0.270962 0.545362 0.665032        True
       baseline          0.501149         0.012641          0.478885         0.014929             1.492177 0.433534 0.271604 0.545170 0.665234        True
      C=0.01_l2          0.500794         0.017039          0.477198  

## 9. Registro de resultados (convención US-03)

Cada ejecución real se añade (append-only) a `results/logistic_regression.csv` con las columnas de la convención: `run_timestamp_utc, seed, model, config, partition, accuracy, macro_f1, f1_A..f1_D, notes`. Nunca se editan ni borran filas anteriores. Mientras US-05 fase 2 no esté integrada en `develop`, **todas las filas nuevas llevan la marca PROVISIONAL en `notes`**.

Esta ejecución de D-015 conserva las 20 filas anteriores y añade **8 filas nuevas**: una fila `cv5fold` por configuración (media de los cinco folds, con `accuracy_std`, `macro_f1_std` y el train - CV accuracy gap en `notes`) y las filas `train_70` y `val_15` de la configuración seleccionada. La fila oficial de `val_15` incluye el gap de D-010 y precision, recall y F1 por clase en `notes` (D-015, ítem 8). No se registra de nuevo el Dummy: sus dos filas de la ejecución previa siguen vigentes.


In [9]:
from datetime import datetime, timezone

RESULTS_CSV = Path("results") / "logistic_regression.csv"
PROVISIONAL = "PROVISIONAL (US-05 fase 2 no integrada en develop)"


def append_row(config: str, partition: str, accuracy: float, macro_f1: float, f1_values, notes: str = "") -> None:
    """Añade una fila de métricas de una ejecución real (append-only)."""
    row = pd.DataFrame(
        [
            {
                "run_timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "seed": RANDOM_STATE,
                "model": "logistic_regression",
                "config": config,
                "partition": partition,
                "accuracy": accuracy,
                "macro_f1": macro_f1,
                "f1_A": f1_values["A"],
                "f1_B": f1_values["B"],
                "f1_C": f1_values["C"],
                "f1_D": f1_values["D"],
                "notes": notes,
            }
        ]
    )
    row.to_csv(RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False)


def record(config: str, partition: str, y_true, y_pred, notes: str = "") -> None:
    """Registra las métricas de una predicción (train_70 o val_15)."""
    _, _, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASSES, zero_division=0
    )
    append_row(
        config,
        partition,
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
        dict(zip(CLASSES, f1)),
        notes,
    )


# --- 1) Una fila cv5fold por configuración: media y desviación típica de los
#          cinco scores de fold (D-015, ítems 2 y 8).
for row in cv_summary.to_dict("records"):
    cv_note = (
        f"{PROVISIONAL} | D-015 cv5fold media de 5 scores de fold; "
        f"accuracy_std={row['cv_accuracy_std']:.6f}; "
        f"macro_f1_std={row['cv_macro_f1_std']:.6f}; "
        f"train_cv_acc_gap={row['train_cv_acc_gap_pp']:+.2f} pp"
    )
    append_row(
        row["config"],
        "cv5fold",
        row["cv_accuracy_mean"],
        row["cv_macro_f1_mean"],
        {label: row[f"cv_f1_{label}"] for label in CLASSES},
        cv_note,
    )

# --- 2) Evaluación oficial de la configuración seleccionada por D-015.
selected_cfg = next(
    cfg for cfg in CANDIDATE_CONFIGS if cfg["config"] == selected_cv["config"]
)
selected_pipe = build_logreg_pipeline(
    C=selected_cfg["C"],
    penalty=selected_cfg["penalty"],
    solver=selected_cfg["solver"],
    class_weight=selected_cfg["class_weight"],
)
selected_pipe.fit(X_train, y_train)
pred_train = selected_pipe.predict(X_train)
pred_val = selected_pipe.predict(X_val)
gap_pp = (accuracy_score(y_train, pred_train) - accuracy_score(y_val, pred_val)) * 100
precision, recall, f1, support = precision_recall_fscore_support(
    y_val, pred_val, labels=CLASSES, zero_division=0
)
val_classes = "; ".join(
    f"{label}={precision[i]:.4f}/{recall[i]:.4f}/{f1[i]:.4f}"
    for i, label in enumerate(CLASSES)
)

selection_note = (
    f"{PROVISIONAL} | D-015 seleccionada: mayor macro-F1 CV entre admisibles "
    f"(CV macro-F1={selected_cv['cv_macro_f1_mean']:.6f}, "
    f"std={selected_cv['cv_macro_f1_std']:.6f}, "
    f"train-CV gap={selected_cv['train_cv_acc_gap_pp']:+.2f} pp) | "
    f"gap D-010 {gap_pp:+.2f} pp "
    + ("CUMPLE" if gap_pp < 5.0 else "NO CUMPLE")
    + f" | precision/recall/F1 val: {val_classes}"
)
record(
    selected_cfg["config"],
    "train_70",
    y_train,
    pred_train,
    PROVISIONAL + " | D-015 selected configuration; official check on the val_15 row of this run",
)
record(selected_cfg["config"], "val_15", y_val, pred_val, selection_note)

print(f"Filas añadidas a {RESULTS_CSV}: 8 (6 cv5fold + train_70 y val_15 de la config seleccionada)")
print(
    f"Seleccionada: {selected_cfg['config']} | val acc={accuracy_score(y_val, pred_val):.4f} | "
    f"val macro-F1={f1_score(y_val, pred_val, average='macro'):.4f} | gap={gap_pp:+.2f} pp"
)
print("Precision/recall/F1 en val_15:", val_classes)


Filas añadidas a results\logistic_regression.csv: 8 (6 cv5fold + train_70 y val_15 de la config seleccionada)
Seleccionada: C=1_l2_balanced | val acc=0.4959 | val macro-F1=0.4786 | gap=+1.64 pp
Precision/recall/F1 en val_15: A=0.3949/0.4189/0.4066; B=0.3493/0.2616/0.2992; C=0.4908/0.5424/0.5153; D=0.6731/0.7147/0.6933


## 10. Análisis de resultados y conclusiones

**Ejecutado el 2026-11-17.** Todas las cifras provienen de ejecuciones reales registradas en `results/logistic_regression.csv`. Esta ejecución aplica D-015: conserva las 20 filas de la ejecución previa y añade 8 filas nuevas (6 `cv5fold` con media y desviación típica de los cinco folds, más `train_70` y `val_15` de la configuración seleccionada). Semilla `RANDOM_STATE = 42` en todas las filas.

**Baseline (referencia mínima D-010).** `DummyClassifier(strategy="most_frequent")`: en `val_15` accuracy 0.2810 / macro-F1 0.1097 (predice siempre D; f1_D ≈ 0.4387).

**Dimensiones y nombres.** El preprocesamiento con D-011 + D-014 conectados por parámetros produce **28 columnas** ajustado en `train_70` (3 numéricas + 2 indicadores de nulos + 2 binarias + 1 `Gender` + 10 `Profession` con `Unknown` + 7 `Var_1` + 3 `Spending_Score`). El `ColumnTransformer` interno del imputador nominal usa `verbose_feature_names_out=False`, así que las columnas se llaman `Gender_Male`, `Profession_*`, `Var_1_Cat_*`, etc., sin los prefijos `mode__`/`unknown__`; los nombres coinciden con los de los pipelines de random forest y XGBoost para poder agrupar importancias en US-12.

**Comparación CV-5 (D-015) y evaluación oficial:**

| config | CV macro-F1 media ± std | train - CV acc gap | val acc | val macro-F1 | gap D-010 |
|---|---:|---:|---:|---:|---:|
| **C=1_l2_balanced (seleccionada)** | **0.4857 ± 0.0101** | +1.39 pp | 0.4959 | 0.4786 | +1.64 pp ✓ |
| C=0.1_l2 | 0.4795 ± 0.0173 | +1.38 pp | 0.5066 | 0.4806 | +0.55 pp ✓ |
| C=1_l1_saga | 0.4790 ± 0.0168 | +1.42 pp | 0.5066 | 0.4804 | +0.73 pp ✓ |
| C=10_l2 | 0.4790 ± 0.0150 | +1.46 pp | 0.5050 | 0.4790 | +0.75 pp ✓ |
| baseline (C=1_l2) | 0.4789 ± 0.0149 | +1.49 pp | 0.5058 | 0.4797 | +0.71 pp ✓ |
| C=0.01_l2 | 0.4772 ± 0.0195 | +1.49 pp | 0.5025 | 0.4757 | +1.14 pp ✓ |

- **Todas** las configuraciones son admisibles en CV (train - CV accuracy gap < 5 pp) y superan con claridad al Dummy en `val_15` (macro-F1 0.1097).
- **Selección D-015:** la mejor media de macro-F1 en CV es `C=1_l2_balanced` (0.485673) con desviación típica 0.010076; su error estándar es 0.004506 y el umbral de empate 0.481166. Ninguna otra configuración alcanza ese umbral (la siguiente es `C=0.1_l2`, 0.479499), así que el grupo empatado contiene solo a `C=1_l2_balanced` y se selecciona esa configuración.
- **La selección no usa `val_15`.** `C=0.1_l2` tiene el mayor macro-F1 en `val_15` (0.4806), pero eso no decide la configuración con D-015; de hecho, `C=0.1_l2` es segunda en macro-F1 de CV (0.4795), no primera. La frase de la versión anterior que la situaba "también primera" en CV era incorrecta y queda corregida.
- **Comprobación oficial:** `C=1_l2_balanced` obtiene en `val_15` accuracy 0.495868, macro-F1 0.478583 y gap de accuracy +1.64 pp, por debajo del límite de 5 pp de D-010. Las métricas son peores que las de `C=0.1_l2` en `val_15`, pero D-015 prioriza la estimación CV para elegir y reserva `val_15` para comprobar que la elección no falla el límite.

**Detalle por clase de la configuración seleccionada en `val_15`:**

| clase | precision | recall | F1 |
|---|---:|---:|---:|
| A | 0.3949 | 0.4189 | 0.4066 |
| B | 0.3493 | 0.2616 | 0.2992 |
| C | 0.4908 | 0.5424 | 0.5153 |
| D | 0.6731 | 0.7147 | 0.6933 |

La clase B sigue siendo la más difícil (F1 0.2992), aunque `class_weight="balanced"` mejora su F1 respecto a las configuraciones sin balanceo (≈0.26–0.28); A, C y D quedan por debajo de `C=0.1_l2`. El detalle completo está en la fila `val_15` de `C=1_l2_balanced` del CSV (`notes`).

**Limitaciones pendientes:**
1. **PROVISIONAL:** US-05 fase 2 no está integrada en `develop`. Este notebook conecta D-011/D-014 por parámetros (D-014, ítem 5) y mide 28 columnas, pero las métricas deben re-ejecutarse cuando la fase 2 aterrice (D-013, Consequences) antes de la comparación final de US-11.
2. scikit-learn 1.9.1 marca `penalty` como deprecado (se elimina en 1.10; migración futura a `l1_ratio`). Con el entorno fijado en `requirements.txt` los resultados son reproducibles.
3. La comparación definitiva con los otros modelos ocurre en US-11 sobre la partición compartida (D-013); estos números no son comparables de forma definitiva hasta entonces.
4. `test_15.csv` no se carga ni se usa en ningún punto (D-012); se abrirá una sola vez en US-12 con el modelo congelado.
